# MixesDB search → tracklists

Enter a search word (e.g. `hawtin`). This notebook:
1. Searches MixesDB for mixes whose **title** contains the word (via `mixesdb_scraper.py`)
2. Fetches the tracklist for each matching mix (via `mixesdb_tracklist.py`)
3. Prints the tracklist for every mix that has one, and skips those without
4. Optionally saves all tracklists to a single long table in one XLSX sheet

In [44]:
import sys, os
sys.path.insert(0, os.getcwd())

from mixesdb import search, get_tracklist
import pandas as pd

## 1. Enter the search word

In [ ]:
query = 'Nick Curly'
query = 'Sonja Moonear'
query = 'Barem'
query = 'Karotte'
query = 'Dan Ghenacia'
query = 'Tania Vulcano'
query = 'Music on'  # search query; change to whatever you want
max_mixes = 2000  # how many matching mixes to fetch tracklists for; raise if you want more

print(f"Searching MixesDB for '{query}'...")

Searching MixesDB for 'Boiler Room'...


In [46]:
query

'Boiler Room'

## 2. Run the search (title must contain the word)

In [47]:
results = search(query, limit=max_mixes, strict=True)

print(f"Found {len(results)} mix(es) with '{query}' in the title.\n")
pd.DataFrame(results)[["title", "url"]] if results else print("No matching mixes found.")

Found 2000 mix(es) with 'Boiler Room' in the title.



,title,url
0,2014-11-25 - Karenn (Live PA) @ Boiler Room Lo...,https://www.mixesdb.com/w/2014-11-25_-_Karenn_...
1,2011-08-23 - Kassem Mosse (Live PA) @ Boiler R...,https://www.mixesdb.com/w/2011-08-23_-_Kassem_...
2,2015-03-10 - Lory D (Live PA) @ Boiler Room x ...,https://www.mixesdb.com/w/2015-03-10_-_Lory_D_...
3,2012-12-12 - Sven Väth @ Boiler Room Berlin 01...,https://www.mixesdb.com/w/2012-12-12_-_Sven_V%...
4,2011-08-23 - Redshape (Live PA) @ Boiler Room ...,https://www.mixesdb.com/w/2011-08-23_-_Redshap...
...,...,...
1995,2019-03-16 - Eris Drew @ Boiler Room Bogotá,https://www.mixesdb.com/w/2019-03-16_-_Eris_Dr...
1996,"2014-07-24 - Frank & Tony @ SKYY Stream, Boile...",https://www.mixesdb.com/w/2014-07-24_-_Frank_&...
1997,"2018-04-19 - Chaos In The CBD @ True Music, Bo...",https://www.mixesdb.com/w/2018-04-19_-_Chaos_I...
1998,2018-01-28 - Sol Ortega @ Boiler Room Buenos A...,https://www.mixesdb.com/w/2018-01-28_-_Sol_Ort...


## 3. Fetch and display the tracklist for each mix

In [48]:
all_tracks = []  # flat rows for the CSV export: mix_title, mix_url, part, position, timestamp, artist, title, raw

for r in results:
    print(f"\n{'='*80}\n{r['title']}\n{r['url']}\n{'='*80}")

    try:
        tracks = get_tracklist(r["url"])
    except Exception as e:
        print(f"  Could not fetch tracklist: {e}")
        continue

    if not tracks:
        print("  No tracklist found on this page.")
        continue

    current_part = None
    for t in tracks:
        if t["part"] != current_part:
            current_part = t["part"]
            if current_part:
                print(f"\n-- {current_part} --")
        ts = f"[{t['timestamp']}] " if t["timestamp"] else ""
        line = f"{t['artist']} - {t['title']}" if t["artist"] else t["raw"]
        print(f"{t['position']:>3}. {ts}{line}")

        all_tracks.append({
            "mix_title": r["title"],
            "mix_url": r["url"],
            **t,
        })

# print(f"\n\nDone. Collected {len(all_tracks)} tracks across {len(results)} mix(es).")


2014-11-25 - Karenn (Live PA) @ Boiler Room London
https://www.mixesdb.com/w/2014-11-25_-_Karenn_(Live_PA)_@_Boiler_Room_London
  No tracklist found on this page.

2011-08-23 - Kassem Mosse (Live PA) @ Boiler Room Berlin 001
https://www.mixesdb.com/w/2011-08-23_-_Kassem_Mosse_(Live_PA)_@_Boiler_Room_Berlin_001
  No tracklist found on this page.

2015-03-10 - Lory D (Live PA) @ Boiler Room x Numbers, Glasgow
https://www.mixesdb.com/w/2015-03-10_-_Lory_D_(Live_PA)_@_Boiler_Room_x_Numbers,_Glasgow
  1. Lory D - Tr 3
  2. Lory D - ob.bx-n [Monkeytown - MTR089]
  3. Lory D - Bassbam [Numbers - NMBRS 41]
  4. Lory D - Copeland_Remix
  5. Lory D - Sq11 [Numbers - NMBRS 41]
  6. Lory D - Adronix [Numbers - NMBRS13]
  7. Lory D - grn-HF [Numbers - NMBRS38]
  8. Lory D - 909Trap
  9. Lory D - Ghill [Wireblock - WB001]
 10. Lory D - Acidrum
 11. Lory D - Acid-CLX22 [Numbers - NMBRS44]
 12. Lory D - Acidattak2 [Numbers - NMBRS23]
 13. Lory D - Xacx01
 14. Lory D - Happy90Ties
 15. Lory D - Monkey

## 4. (optional) Save all tracklists to a single XLSX sheet

In [49]:
if all_tracks:
    out_path = f"{query.replace(' ', '_')}_tracklists.xlsx"
    df = pd.DataFrame(all_tracks)
    cols = ["mix_title", "mix_url", "part", "position", "timestamp", "artist", "title", "raw"]

    df[cols].to_excel(out_path, sheet_name="tracklists", index=False, engine="openpyxl")

    print(f"Saved {len(df)} rows across {df['mix_title'].nunique()} tracklist(s) to {out_path}")
else:
    print("Nothing to save — no tracklists were found.")

Saved 11638 rows across 668 tracklist(s) to Boiler_Room_tracklists.xlsx
